# Notebook 02 — File Identification and Metadata Mapping
## Parkinson_Gait_CIGTNet

**Research Title:** A Clinically-Informed Multiscale Graph-Transformer Framework for Explainable Parkinsonian Gait Severity Assessment from Multisensor Time-Series Data

---

### Core Principle & Scope Division
* **Notebook 01** established the empirical discovery of the raw dataset: file layout, container formats (Excel binary `.xls` named `.csv`), runtime-discovered total channels (`n_linear_features` linear + `n_angular_features` angular), sequence lengths, and clinical scales in `PDGinfo.xlsx`.
* **Notebook 02 (This Notebook)** serves **only** as the lightweight deterministic file-mapping and validation layer. It answers:
  > *Which exact raw files should be used together for downstream processing, and can they be identified deterministically?*
* **Notebook 03** will consume the validated trial index produced here to perform physical stream synchronization and merging.

### What is Explicitly Excluded (Handled in Notebook 01 or Later)
* **NO** repeated dataset exploration or subject inventory summaries.
* **NO** Notebook 01 subject metadata extraction or clinical scale analysis (H&Y / UPDRS).
* **NO** column-level schema discovery or channel count inspection.
* **NO** sequence-length distribution or timestamp resampling analysis.
* **NO** data normalization, GSN, TBSN, windowing, or model training.
* **NO** modification to `data/raw/` (treated as strictly read-only).


## Scope guard — Notebook 02 is intentionally lightweight

Notebook 01 is authoritative for dataset exploration, schema discovery, subject/condition discovery, clinical discovery, sequence characteristics, and broad data-quality exploration.

This notebook does **not** repeat those analyses. It only converts the validated Notebook 01 inventory into a deterministic kinematic file/trial mapping for Notebook 03, checks file accessibility, and flags pairing conflicts.

**Downstream contract:** `data/metadata/trial_metadata_index.csv`


## Section 2: Environment & Dynamic Project Root Resolution

The project root is resolved dynamically by traversing upward for configuration files. **No machine-specific absolute paths are used.**


In [1]:
import sys
import os
import re
import json
import datetime
import warnings
import platform
import yaml
import numpy as np
import pandas as pd
import xlrd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Dynamic project root discovery (NO HARDCODED PATHS)
# ---------------------------------------------------------------------------
def find_project_root():
    curr = os.path.abspath(os.getcwd())
    for _ in range(6):
        if (
            os.path.isfile(os.path.join(curr, "configs", "config.yaml"))
            and os.path.isfile(os.path.join(curr, "configs", "data_config.yaml"))
            and os.path.isdir(os.path.join(curr, "src"))
            and os.path.isdir(os.path.join(curr, "data"))
        ):
            return curr
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    raise RuntimeError("Could not resolve project root containing configs/ and src/")

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Load master configurations
with open(os.path.join(PROJECT_ROOT, "configs", "config.yaml"), "r") as f:
    cfg = yaml.safe_load(f)
with open(os.path.join(PROJECT_ROOT, "configs", "data_config.yaml"), "r") as f:
    data_cfg = yaml.safe_load(f)

RANDOM_SEED = cfg["reproducibility"]["global_seed"]
np.random.seed(RANDOM_SEED)

RAW_DIR = os.path.join(PROJECT_ROOT, data_cfg["raw_dataset"]["source_dir"])
RESULTS_DIR = os.path.join(PROJECT_ROOT, cfg["paths"]["results_dir"], "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, cfg["paths"]["figures_dir"], "preprocessing")
METADATA_DIR = os.path.join(PROJECT_ROOT, cfg["paths"]["metadata_dir"])

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

# Pre-execution snapshot of raw directory to ensure immutability
def snapshot_raw_dir(root_dir):
    snap = {}
    for r, dirs, files in os.walk(root_dir):
        for f in files:
            p = os.path.join(r, f)
            rel = os.path.relpath(p, root_dir)
            try:
                st = os.stat(p)
                snap[rel] = (st.st_size, st.st_mtime)
            except OSError:
                pass
    return snap

PRE_SNAPSHOT = snapshot_raw_dir(RAW_DIR)

print(f"Project Root:      {PROJECT_ROOT}")
print(f"Python Version:    {platform.python_version()}")
print(f"Raw Files Count:   {len(PRE_SNAPSHOT)} (snapshot recorded)")


Project Root:      c:\Users\lab\Minor_project\Parkinson_gait
Python Version:    3.11.0
Raw Files Count:   4831 (snapshot recorded)


## Section 3: Ingestion of Notebook 01 Authoritative Artifacts

We read the authoritative inventories established by Notebook 01 rather than repeating full dataset exploration.


In [2]:
manifest_path = os.path.join(METADATA_DIR, "dataset_manifest.json")
file_inv_path = os.path.join(RESULTS_DIR, "file_inventory.csv")
consistency_path = os.path.join(RESULTS_DIR, "trial_consistency_inventory.csv")
pairing_inv_path = os.path.join(RESULTS_DIR, "modality_pairing_inventory.csv")

assert os.path.isfile(manifest_path), f"Missing required Notebook 01 artifact: {manifest_path}"
assert os.path.isfile(file_inv_path), f"Missing required Notebook 01 artifact: {file_inv_path}"

with open(manifest_path, "r", encoding="utf-8") as fp:
    nb01_manifest = json.load(fp)

nb01_files_df = pd.read_csv(file_inv_path)
print(f"Loaded Notebook 01 manifest: project={nb01_manifest.get('project')}, timestamp={nb01_manifest.get('analysis_timestamp')}")
print(f"Loaded Notebook 01 file inventory: {len(nb01_files_df)} total files cataloged")


Loaded Notebook 01 manifest: project=Parkinson_gait, timestamp=None
Loaded Notebook 01 file inventory: 4515 total files cataloged


## Section 4: Build Lightweight Kinematic File Index

We extract only the linear and angular kinematic walking files required for downstream merging.
A deterministic `file_id` is assigned to each file.


In [3]:
kin_mask = nb01_files_df["modality"].isin(["linear_kinematics", "angular_kinematics"])
kin_files_df = nb01_files_df[kin_mask].copy().sort_values(["subject_id", "medication_state", "trial_id", "modality"]).reset_index(drop=True)

# Assign deterministic, sequential file IDs
kin_files_df["file_id"] = [f"KIN_{i+1:04d}" for i in range(len(kin_files_df))]
kin_files_df["condition"] = kin_files_df["medication_state"].str.lower()
def safe_trial_id(x):
    if pd.isna(x):
        return "UNKNOWN"
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return "INVALID"

kin_files_df["trial_id"] = kin_files_df["trial_id"].apply(safe_trial_id)

print(f"Identified kinematic files for downstream pipeline: {len(kin_files_df)}")
print(kin_files_df["modality"].value_counts().to_string())


Identified kinematic files for downstream pipeline: 1768
modality
angular_kinematics    884
linear_kinematics     884


## Section 5: Validate File Existence and Low-Level Readability

For every identified kinematic file, we verify:
1. Physical existence on disk (`full_path`).
2. Readability via `xlrd` (verifying header at row 4 and valid data rows).


In [4]:
exists_flags = []
readable_flags = []
read_errors = []

for _, row in kin_files_df.iterrows():
    p = row["full_path"]
    f_exists = os.path.isfile(p)
    exists_flags.append(f_exists)
    
    if not f_exists:
        readable_flags.append(False)
        read_errors.append("FILE_NOT_FOUND")
        continue
        
    try:
        wb = xlrd.open_workbook(p, on_demand=True)
        sheet = wb.sheet_by_index(0)
        # Verify sheet has header row (row index 4) and at least one data row
        if sheet.nrows > 5 and sheet.ncols > 2:
            readable_flags.append(True)
            read_errors.append(None)
        else:
            readable_flags.append(False)
            read_errors.append("INVALID_ROW_STRUCTURE")
    except Exception as e:
        readable_flags.append(False)
        read_errors.append(str(e)[:100])

kin_files_df["exists"] = exists_flags
kin_files_df["readable"] = readable_flags
kin_files_df["read_error"] = read_errors

print(f"File Existence Check:   {kin_files_df['exists'].sum()} / {len(kin_files_df)} exist (100.0%)")
print(f"File Readability Check: {kin_files_df['readable'].sum()} / {len(kin_files_df)} readable (100.0%)")


File Existence Check:   1768 / 1768 exist (100.0%)
File Readability Check: 1768 / 1768 readable (100.0%)


## Section 6: Construct Canonical Trial Keys

Every kinematic trial is assigned a canonical identifier:
$$\text{trial\_key} = \text{subject\_id} + \text{"\_"} + \text{condition} + \text{"\_walk\_"} + \text{trial\_id}$$


In [5]:
kin_files_df["trial_key"] = (
    kin_files_df["subject_id"].astype(str) + "_" +
    kin_files_df["condition"].astype(str) + "_walk_" +
    kin_files_df["trial_id"].astype(str)
)

print(f"Sample canonical trial keys:")
print(kin_files_df[["file_id", "trial_key", "modality"]].head(6).to_string(index=False))


Sample canonical trial keys:
 file_id        trial_key           modality
KIN_0001 SUB01_off_walk_1 angular_kinematics
KIN_0002 SUB01_off_walk_1  linear_kinematics
KIN_0003 SUB01_off_walk_2 angular_kinematics
KIN_0004 SUB01_off_walk_2  linear_kinematics
KIN_0005 SUB01_off_walk_3 angular_kinematics
KIN_0006 SUB01_off_walk_3  linear_kinematics


## Section 7: Validate Linear / Angular Kinematic Pairing

We cross-match linear and angular streams by canonical `trial_key` and assign pairing status:
* `PAIRED`: Exactly one linear and one angular file exist.
* `LINEAR_ONLY`: Linear exists without angular counterpart.
* `ANGULAR_ONLY`: Angular exists without linear counterpart.
* `DUPLICATE_LINEAR` / `DUPLICATE_ANGULAR`: Collision on stream ID.
* `AMBIGUOUS`: Conflicting metadata.


In [6]:
linear_subset = kin_files_df[kin_files_df["modality"] == "linear_kinematics"]
angular_subset = kin_files_df[kin_files_df["modality"] == "angular_kinematics"]

all_trial_keys = sorted(set(kin_files_df["trial_key"].unique()))

lin_by_key = {k: v for k, v in linear_subset.groupby("trial_key")}
ang_by_key = {k: v for k, v in angular_subset.groupby("trial_key")}

pairing_records = []

for k in all_trial_keys:
    l_matches = lin_by_key.get(k, pd.DataFrame())
    a_matches = ang_by_key.get(k, pd.DataFrame())
    
    n_lin = len(l_matches)
    n_ang = len(a_matches)
    
    l_row = l_matches.iloc[0] if n_lin > 0 else None
    a_row = a_matches.iloc[0] if n_ang > 0 else None
    
    sid = l_row["subject_id"] if l_row is not None else a_row["subject_id"]
    cond = l_row["condition"] if l_row is not None else a_row["condition"]
    tid = l_row["trial_id"] if l_row is not None else a_row["trial_id"]
    
    # Evaluate status
    if n_lin == 1 and n_ang == 1:
        p_status = "PAIRED"
    elif n_lin > 1:
        p_status = "DUPLICATE_LINEAR"
    elif n_ang > 1:
        p_status = "DUPLICATE_ANGULAR"
    elif n_lin == 1 and n_ang == 0:
        p_status = "LINEAR_ONLY"
    elif n_lin == 0 and n_ang == 1:
        p_status = "ANGULAR_ONLY"
    else:
        p_status = "AMBIGUOUS"
        
    pairing_records.append({
        "trial_key": k,
        "subject_id": sid,
        "condition": cond,
        "trial_id": tid,
        "linear_file_id": l_row["file_id"] if l_row is not None else None,
        "angular_file_id": a_row["file_id"] if a_row is not None else None,
        "linear_path": l_row["relative_path"] if l_row is not None else None,
        "angular_path": a_row["relative_path"] if a_row is not None else None,
        "linear_available": n_lin > 0,
        "angular_available": n_ang > 0,
        "pairing_status": p_status,
        "duplicate_flag": (n_lin > 1 or n_ang > 1),
        "readability_status": "READABLE" if ((l_row is None or l_row["readable"]) and (a_row is None or a_row["readable"])) else "UNREADABLE",
        "metadata_conflict_flag": False,
        "validation_status": "VALIDATED" if p_status == "PAIRED" else "REQUIRES_REVIEW",
    })

trial_meta_df = pd.DataFrame(pairing_records)

print("Pairing Status Breakdown:")
print(trial_meta_df["pairing_status"].value_counts().to_string())


Pairing Status Breakdown:
pairing_status
PAIRED    884


## Section 8: Detect Duplicate or Conflicting Mappings

We verify that no duplicate metadata keys exist and check consistency between directory path tokens and filename tokens.


In [7]:
# Check for metadata duplicates
meta_dups = kin_files_df.duplicated(subset=["subject_id", "condition", "trial_id", "modality"], keep=False)
kin_files_df["duplicate_flag"] = meta_dups

# Check path-vs-filename consistency
conflicts = []
for _, row in kin_files_df.iterrows():
    p_dir = row["parent_directory"]
    fname = row["filename"]
    
    m_dir = re.match(r"^(SUB\d+)_(on|off)$", p_dir, re.I)
    m_file = re.match(r"^(SUB\d+)_(on|off)_walk_(\d+)", fname, re.I)
    
    if m_dir and m_file:
        s_d, c_d = m_dir.group(1).upper(), m_dir.group(2).lower()
        s_f, c_f, t_f = m_file.group(1).upper(), m_file.group(2).lower(), str(int(m_file.group(3)))
        if s_d != s_f or c_d != c_f:
            conflicts.append({
                "file_id": row["file_id"],
                "filename": fname,
                "dir_tokens": f"{s_d}_{c_d}",
                "file_tokens": f"{s_f}_{c_f}",
            })

conflict_dict = {c["file_id"]: c for c in conflicts}
kin_files_df["metadata_conflict_flag"] = kin_files_df["file_id"].isin(conflict_dict.keys())
kin_files_df["metadata_conflict_fields"] = kin_files_df["file_id"].apply(lambda x: "dir_vs_filename" if x in conflict_dict else None)
kin_files_df["metadata_conflict_reason"] = kin_files_df["file_id"].apply(lambda x: f"Dir ({conflict_dict[x]['dir_tokens']}) vs File ({conflict_dict[x]['file_tokens']})" if x in conflict_dict else None)

# Update trial_meta_df
trial_meta_df["metadata_conflict_flag"] = trial_meta_df.apply(
    lambda r: (r["linear_file_id"] in conflict_dict) or (r["angular_file_id"] in conflict_dict), axis=1
)

print(f"Metadata key duplicates: {meta_dups.sum()}")
print(f"Metadata path-vs-filename conflicts: {len(conflicts)}")


Metadata key duplicates: 0
Metadata path-vs-filename conflicts: 0


## Section 9: Problem File and Trial Report

Any trial or file that fails pairing or readability is cataloged in `results/tables/file_identification_problems.csv`.


In [8]:
problem_records = []

# Check pairing problems
unpaired_trials = trial_meta_df[trial_meta_df["pairing_status"] != "PAIRED"]
for _, row in unpaired_trials.iterrows():
    problem_records.append({
        "trial_key": row["trial_key"],
        "subject_id": row["subject_id"],
        "condition": row["condition"],
        "trial_id": row["trial_id"],
        "problem_type": row["pairing_status"],
        "description": f"Trial pairing status is {row['pairing_status']}",
    })

# Check unreadable files
unreadable_files = kin_files_df[~kin_files_df["readable"]]
for _, row in unreadable_files.iterrows():
    problem_records.append({
        "trial_key": row["trial_key"],
        "subject_id": row["subject_id"],
        "condition": row["condition"],
        "trial_id": row["trial_id"],
        "problem_type": "UNREADABLE_FILE",
        "description": f"File {row['filename']} could not be parsed: {row['read_error']}",
    })

problem_df = pd.DataFrame(problem_records)
if len(problem_df) == 0:
    problem_df = pd.DataFrame(columns=["trial_key", "subject_id", "condition", "trial_id", "problem_type", "description"])
    print("No identification problems detected. All files and trials are valid and paired.")
else:
    print(f"Identified {len(problem_df)} problems requiring attention:")
    print(problem_df.to_string(index=False))

problem_df.to_csv(os.path.join(RESULTS_DIR, "file_identification_problems.csv"), index=False)
print(f"Saved -> results/tables/file_identification_problems.csv")


No identification problems detected. All files and trials are valid and paired.
Saved -> results/tables/file_identification_problems.csv


## Section 10: Generate Authoritative Metadata Indexes

We export:
1. `data/metadata/trial_metadata_index.csv`: Authoritative trial pairing contract for Notebook 03.
2. `data/metadata/file_metadata_index.csv`: Lightweight file index.
3. `results/tables/modality_pairing_validation.csv`: Quantitative pairing summary.


In [9]:
# 3. Correct validation status
kin_files_df["validation_status"] = kin_files_df.apply(
    lambda row: "INVALID" if not row["readable"] or row["trial_id"] in ["INVALID", "UNKNOWN"] else (
        "REQUIRES_REVIEW" if row["metadata_conflict_flag"] or row["duplicate_flag"] else "VALIDATED"
    ), axis=1
)

trial_meta_df["validation_status"] = trial_meta_df.apply(
    lambda row: "INVALID" if row["readability_status"] == "UNREADABLE" or row["trial_id"] in ["INVALID", "UNKNOWN"] else (
        "REQUIRES_REVIEW" if row["pairing_status"] != "PAIRED" or row["metadata_conflict_flag"] or row["duplicate_flag"] else "VALIDATED"
    ), axis=1
)

# 5. Create explicit merge-eligibility gate
trial_meta_df["eligible_for_merge"] = (
    (trial_meta_df["pairing_status"] == "PAIRED") & 
    (trial_meta_df["readability_status"] == "READABLE") & 
    (~trial_meta_df["metadata_conflict_flag"]) & 
    (~trial_meta_df["duplicate_flag"]) & 
    (trial_meta_df["trial_id"] != "INVALID") &
    (trial_meta_df["trial_id"] != "UNKNOWN")
)

# 1. Authoritative Trial Metadata Index
trial_cols = [
    "trial_key", "subject_id", "condition", "trial_id",
    "linear_file_id", "angular_file_id", "linear_path", "angular_path",
    "linear_available", "angular_available", "pairing_status",
    "duplicate_flag", "readability_status", "metadata_conflict_flag", 
    "validation_status", "eligible_for_merge"
]
trial_meta_df = trial_meta_df[trial_cols].copy()
trial_meta_df.to_csv(os.path.join(METADATA_DIR, "trial_metadata_index.csv"), index=False)

# 2. Lightweight File Metadata Index
file_cols = [
    "file_id", "relative_path", "filename", "subject_id", "condition",
    "trial_id", "modality", "exists", "readable", "duplicate_flag",
    "metadata_conflict_flag", "metadata_conflict_fields", "metadata_conflict_reason", "validation_status"
]
light_file_df = kin_files_df[file_cols].copy()
light_file_df.to_csv(os.path.join(METADATA_DIR, "file_metadata_index.csv"), index=False)

# 3. Modality Pairing Validation Report / Audit Summary
val_report = [{
    "total_files": len(kin_files_df),
    "linear_files": int((kin_files_df["modality"] == "linear_kinematics").sum()),
    "angular_files": int((kin_files_df["modality"] == "angular_kinematics").sum()),
    "paired_trials": int((trial_meta_df["pairing_status"] == "PAIRED").sum()),
    "linear_only_trials": int((trial_meta_df["pairing_status"] == "LINEAR_ONLY").sum()),
    "angular_only_trials": int((trial_meta_df["pairing_status"] == "ANGULAR_ONLY").sum()),
    "duplicate_trial_keys": int(trial_meta_df["pairing_status"].isin(["DUPLICATE_LINEAR", "DUPLICATE_ANGULAR"]).sum()),
    "ambiguous_trial_keys": int((trial_meta_df["pairing_status"] == "AMBIGUOUS").sum()),
    "metadata_conflicts": int(kin_files_df["metadata_conflict_flag"].sum()),
    "validated_files": int((kin_files_df["validation_status"] == "VALIDATED").sum()),
    "files_requiring_review": int((kin_files_df["validation_status"] == "REQUIRES_REVIEW").sum()),
    "invalid_files": int((kin_files_df["validation_status"] == "INVALID").sum()),
    "eligible_trials_for_notebook_03": int(trial_meta_df["eligible_for_merge"].sum()),
}]
val_report_df = pd.DataFrame(val_report)
val_report_df.to_csv(os.path.join(RESULTS_DIR, "modality_pairing_validation.csv"), index=False)

print(f"Saved -> data/metadata/trial_metadata_index.csv ({len(trial_meta_df)} rows)")
print(f"Saved -> data/metadata/file_metadata_index.csv ({len(light_file_df)} rows)")
print(f"Saved -> results/tables/modality_pairing_validation.csv")
print("\nModality Pairing Validation Summary:")
print(val_report_df.T.to_string())

# 10. Add deterministic assertions
assert len(trial_meta_df[trial_meta_df["eligible_for_merge"]]["trial_key"].unique()) == len(trial_meta_df[trial_meta_df["eligible_for_merge"]]), "Eligible trial keys are not unique"
assert trial_meta_df[trial_meta_df["eligible_for_merge"]]["pairing_status"].eq("PAIRED").all(), "Not all eligible trials are paired"
assert not trial_meta_df[trial_meta_df["eligible_for_merge"]]["metadata_conflict_flag"].any(), "Eligible trials have metadata conflicts"
assert not trial_meta_df[trial_meta_df["eligible_for_merge"]]["duplicate_flag"].any(), "Eligible trials have duplicates"
assert trial_meta_df["pairing_status"].isin(["PAIRED", "LINEAR_ONLY", "ANGULAR_ONLY", "DUPLICATE_LINEAR", "DUPLICATE_ANGULAR", "AMBIGUOUS"]).all(), "Invalid pairing status detected"
assert trial_meta_df["validation_status"].isin(["VALIDATED", "REQUIRES_REVIEW", "INVALID"]).all(), "Invalid validation status detected"


Saved -> data/metadata/trial_metadata_index.csv (884 rows)
Saved -> data/metadata/file_metadata_index.csv (1768 rows)
Saved -> results/tables/modality_pairing_validation.csv

Modality Pairing Validation Summary:
                                    0
total_files                      1768
linear_files                      884
angular_files                     884
paired_trials                     884
linear_only_trials                  0
angular_only_trials                 0
duplicate_trial_keys                0
ambiguous_trial_keys                0
metadata_conflicts                  0
validated_files                  1768
files_requiring_review              0
invalid_files                       0
eligible_trials_for_notebook_03   884


## Section 11: Pairing Diagnostic Figure

We generate a single focused diagnostic figure illustrating pairing completeness across linear and angular streams.


In [10]:
fig, ax = plt.subplots(figsize=(8, 4.5))

status_counts = trial_meta_df["pairing_status"].value_counts()
possible_statuses = ["PAIRED", "LINEAR_ONLY", "ANGULAR_ONLY", "DUPLICATE_LINEAR", "DUPLICATE_ANGULAR", "AMBIGUOUS"]
counts = [status_counts.get(s, 0) for s in possible_statuses]
colors = ["mediumseagreen", "darkorange", "crimson", "purple", "gray"]

bars = ax.bar(possible_statuses, counts, color=colors, edgecolor="white", linewidth=1)
ax.set_title("Linear vs Angular Kinematic Stream Pairing Status", fontsize=12, pad=12)
ax.set_ylabel("Number of Walking Trials", fontsize=10)
ax.set_xlabel("Pairing Status", fontsize=10)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + 10, f"{int(yval)}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylim(0, max(counts) * 1.15 if max(counts) > 0 else 10)
fig.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "modality_pairing.png")
fig.savefig(fig_path, dpi=150)
plt.close(fig)

print(f"Saved diagnostic figure -> {fig_path}")


Saved diagnostic figure -> c:\Users\lab\Minor_project\Parkinson_gait\figures\preprocessing\modality_pairing.png


## Section 12: Raw Data Integrity Post-Check

We verify that the raw data directory remained completely untouched throughout execution.


In [11]:
POST_SNAPSHOT = snapshot_raw_dir(RAW_DIR)

raw_integrity_pass = (
    len(PRE_SNAPSHOT) == len(POST_SNAPSHOT)
    and all(k in POST_SNAPSHOT and POST_SNAPSHOT[k] == PRE_SNAPSHOT[k] for k in PRE_SNAPSHOT)
)

assert raw_integrity_pass, "CRITICAL ERROR: data/raw/ was modified during Notebook 02 execution!"
print(f"Raw Data Integrity Post-Check: PASS (all {len(POST_SNAPSHOT)} files unchanged)")


Raw Data Integrity Post-Check: PASS (all 4831 files unchanged)


## Section 13: Final Validation & Downstream Handoff

### Downstream Contract for Notebook 03 (`03_Linear_Angular_Data_Merging.ipynb`)
Notebook 03 will directly consume:
`data/metadata/trial_metadata_index.csv`
Notebook 03 will iterate through validated pairs (`linear_path`, `angular_path`), load both streams, synchronize frame counts, and merge the `n_linear_features` linear features and `n_angular_features` angular features into the unified runtime-discovered-channel representation.

Notebook 03 does **NOT** need to rediscover files, subjects, trials, or conditions.


In [12]:

# ============================================================
# NOTEBOOK 02 — FINAL VALIDATION & DOWNSTREAM HANDOFF GATE
# Every check is evaluated programmatically. No hardcoded PASS.
# ============================================================
from pathlib import Path as _Path
import os as _os

def _file_ok(p, min_rows=0):
    p = _Path(p)
    if not p.is_file() or p.stat().st_size == 0:
        return False
    if str(p).endswith(".csv"):
        try:
            _df = pd.read_csv(p)
            return len(_df) >= min_rows
        except Exception:
            return False
    return True

VALID_PAIRING_STATUSES = {"PAIRED", "LINEAR_ONLY", "ANGULAR_ONLY", "DUPLICATE_LINEAR", "DUPLICATE_ANGULAR", "AMBIGUOUS"}

# Evaluate every check programmatically
audit_checks = {
    "Notebook 01 artifacts loaded":    "PASS" if len(nb01_files_df) > 0 else "FAIL",
    "Kinematic files identified":      "PASS" if len(kin_files_df) > 0 else "FAIL",
    "File existence check":            "PASS" if kin_files_df["exists"].all() else "FAIL",
    "File readability check":          "PASS" if kin_files_df["readable"].all() else "FAIL",
    "Canonical trial keys":            "PASS" if len(all_trial_keys) > 0 else "FAIL",
    # CORRECTED: Accept any valid pairing status, do not require ALL=PAIRED
    "Pairing status valid":            "PASS" if trial_meta_df["pairing_status"].isin(VALID_PAIRING_STATUSES).all() else "FAIL",
    "PAIRED trial count":              f"INFO: {int((trial_meta_df['pairing_status'] == 'PAIRED').sum())} of {len(trial_meta_df)}",
    "Duplicate mapping check":         "PASS" if not kin_files_df["duplicate_flag"].any() else "WARN",
    "Metadata conflict check":         "PASS" if len(conflicts) == 0 else "WARN",
    "Trial metadata index":            "PASS" if _file_ok(_os.path.join(METADATA_DIR, "trial_metadata_index.csv"), 1) else "FAIL",
    "File metadata index":             "PASS" if _file_ok(_os.path.join(METADATA_DIR, "file_metadata_index.csv"), 1) else "FAIL",
    "Problem report exists":           "PASS" if _file_ok(_os.path.join(RESULTS_DIR, "file_identification_problems.csv")) else "FAIL",
    "Clinical subject index":          "PASS" if _file_ok(_os.path.join(METADATA_DIR, "clinical_subject_index.csv")) else "WARN",
    "Raw feature provenance":          "PASS" if _file_ok(_os.path.join(METADATA_DIR, "raw_feature_provenance.csv"), 1) else "WARN",
    "Raw data integrity":              "PASS" if raw_integrity_pass else "FAIL",
}

# Determine final status — FAIL on any FAIL, WARN on any WARN, PASS otherwise
any_fail = any(v == "FAIL" for v in audit_checks.values())
any_warn = any(v.startswith("WARN") for v in audit_checks.values())

# 6. Meaningful final readiness logic
n_eligible = trial_meta_df["eligible_for_merge"].sum()
if n_eligible == 0 or any_fail:
    final_status = "NOT_READY"
elif n_eligible < len(trial_meta_df) or any_warn:
    final_status = "READY_WITH_WARNINGS"
else:
    final_status = "READY"


print("""
============================================================
NOTEBOOK 02 — FILE MAPPING AUDIT
============================================================""".strip())

for check_name, status in audit_checks.items():
    print(f"  {check_name + ':':<34} {status}")

print(f"\nFINAL STATUS: {final_status}")
print("""
============================================================
Ready to proceed to Notebook 03 (Linear + Angular Data Merging).
============================================================""".strip())

# Save part1 acceptance rows for notebook 02
_nb02_rows = [
    {"requirement": "Notebook 02", "artifact_or_check": k, "status": v if v in {"PASS","FAIL"} else v.split(":")[0], "details": v}
    for k, v in audit_checks.items()
]
_nb02_df = pd.DataFrame(_nb02_rows)
_nb02_df.to_csv(_os.path.join(RESULTS_DIR, "part1_nb02_audit.csv"), index=False)



NOTEBOOK 02 — FILE MAPPING AUDIT
  Notebook 01 artifacts loaded:      PASS
  Kinematic files identified:        PASS
  File existence check:              PASS
  File readability check:            PASS
  Canonical trial keys:              PASS
  Pairing status valid:              PASS
  PAIRED trial count:                INFO: 884 of 884
  Duplicate mapping check:           PASS
  Metadata conflict check:           PASS
  Trial metadata index:              PASS
  File metadata index:               PASS
  Problem report exists:             PASS
  Clinical subject index:            PASS
  Raw feature provenance:            PASS
  Raw data integrity:                PASS

FINAL STATUS: READY
Ready to proceed to Notebook 03 (Linear + Angular Data Merging).


## Final handoff

Notebook 02 is complete only when the authoritative trial mapping is generated and validated.

Notebook 03 must consume `data/metadata/trial_metadata_index.csv` rather than rediscovering file relationships.

Do not perform merging, normalization, GSN, TBSN, scaling, windowing, feature engineering, or model development here.

**STOP after this notebook.**
